# ENRICH-Aerial — GCP-controlled bundle adjustment

Full pipeline on the `nadir-2` sub-block: COLMAP structure from motion, then a
bundle adjustment that adds the surveyed ground control points (and optionally
the camera centres) to COLMAP's own Ceres problem.

The machinery lives in the `pycolmap_ba` package — `frontend` for the pycolmap
stages, `backend` for the extra residuals, `utils` for scoring and export — so
the cells below are the *application*: data, settings, and reading the numbers.
The maths is written up in [`docs/gcp_bundle_adjustment.md`](../docs/gcp_bundle_adjustment.md).


In [ ]:
from pathlib import Path

from pycolmap import logging

from pycolmap_ba import backend, frontend, utils

import enrich_data

# --- paths ------------------------------------------------------------------
DATASET = enrich_data.DATA_ROOT / "full"
image_path = DATASET / "images"
database_path = DATASET / "colmap/database.db"
sfm_free_path = DATASET / "colmap/sparse"  # free network, as the mapper leaves it
sfm_gcp_path = DATASET / "colmap/sparse_gcp"  # after the controlled bundle

# --- what to run ------------------------------------------------------------
# Feature extraction and brute-force matching take a while on CPU, so by default
# an existing database is reused.
FORCE_OVERWRITE_DB = False
# Give the *mapper* the camera centres as pose priors. False leaves the free
# network in an arbitrary frame, which the Sim3 below then fixes.
USE_MAPPER_POSE_PRIORS = False
# Constrain the camera centres in the GCP *bundle*, like Metashape's _GT project
# (all 39 references enabled). The camera-centre accuracy row then becomes a fit
# to data the bundle was given - the honest configuration is False.
USE_CAMERA_PRIORS = True
# Let the controlled bundle free the focal length. Safe here, and only here: the
# control is what breaks the flat-block degeneracy.
REFINE_INTRINSICS = True

logging.set_log_destination(logging.INFO, Path.cwd() / "INFO.log")

## 1. Data

`load()` reads the CSVs as they are; `add_noise()` returns another object of the
same type with a simulated survey applied. `truth` is kept for scoring, `data`
is what the pipeline is allowed to see.

In [ ]:
truth = enrich_data.load(subblock="nadir-2", split_control_check=True)
data = truth  # or: enrich_data.add_noise(truth) to simulate a real survey
print(enrich_data.summary(data))

# --- names used further down ------------------------------------------------
surveyed_xyz = data.marker_xyz
marker_data = data.marker_data  # columns: image_name, marker_name, u, v
camera_centers = data.camera_xyz
marker_names = data.marker_names
control_names = data.control_names
check_names = data.check_names

# --- assumed accuracies (see docs: only the ratios matter) ------------------
tie_points_sigma_px = enrich_data.tie_points_sigma_px
gcp_sigma_px = enrich_data.gcp_sigma_px
gcp_sigma_m = enrich_data.gcp_sigma_m
camera_position_sigma_m = enrich_data.camera_position_sigma_m

print(
    f"survey noise: xy {gcp_sigma_m[0] * 100:.0f} cm, "
    f"z {gcp_sigma_m[2] * 100:.0f} cm; marks {gcp_sigma_px} px"
)
print(f"GCPs: {control_names}")
print(f"CP   {check_names}")

In [ ]:
# Everything the reports need to score the block. Partial truth is fine: each
# field just enables one more row.
gt = utils.GroundTruth(
    camera_xyz=truth.camera_xyz,
    marker_xyz=truth.marker_xyz,
    control_names=truth.control_names,
    check_names=truth.check_names,
    cam_from_world=enrich_data.cam_from_world_all(truth),
    focal_px=enrich_data.FOCAL_PX,
)

## 2. Free network

Features, matches, known intrinsics, incremental mapping. Nothing here knows
about the GCPs yet.

In [ ]:
frontend.extract_and_match(
    database_path,
    image_path,
    # SIMPLE_PINHOLE has a SINGLE focal parameter. With PINHOLE, fx and fy
    # refine independently and on this flat nadir block they diverged to
    # fx=3227.6 / fy=3818.1 - an 18% anisotropic stretch.
    # That is not a similarity, so no alignment can undo it and the whole block
    # ends up ~8.5 m out.
    # The renders have no lens distortion (README), so a distortion-free model
    # with a single focal is the correct choice - and is what Metashape used.
    frontend.FeatureOptions(camera_model="SIMPLE_PINHOLE", max_num_features=8192),
    overwrite=FORCE_OVERWRITE_DB,
)

In [ ]:
# The rendered focal is known exactly, so seed it instead of estimating it.
frontend.seed_intrinsics(
    database_path,
    frontend.CameraIntrinsics(
        width=enrich_data.IMAGE_WIDTH,
        height=enrich_data.IMAGE_HEIGHT,
        params=[enrich_data.FOCAL_PX, *enrich_data.PRINCIPAL_POINT],
    ),
)

In [ ]:
if USE_MAPPER_POSE_PRIORS:
    frontend.write_pose_priors(
        database_path, camera_centers, sigma_m=camera_position_sigma_m
    )

In [ ]:
mappingOptions = frontend.MappingOptions(
    use_prior_position=USE_MAPPER_POSE_PRIORS,
    # A flat nadir block cannot determine focal length from imagery: focal
    # trades against scene depth. Hold the intrinsics at the known values;
    # the GCP bundle can free them later, where the control breaks the
    # degeneracy.
    refine_focal_length=False,
    # COLMAP's own tie-point filter defaults, restated so the tuning surface
    # is visible. All three are NO-OPS on this block - the 80/60 overlap
    # gives strong geometry: measured triangulation angle p1 = 7.4 deg,
    # median 26.8 deg, zero points below 5 deg, max reprojection 3.98 px -
    # but they matter on weaker blocks.
    filter_max_reproj_error=4.0,
    filter_min_tri_angle_deg=1.5,
    ba_local_min_tri_angle_deg=6.0,
)

rec = frontend.run_mapping(
    database_path=database_path,
    image_path=image_path,
    output_path=sfm_free_path,
    options=mappingOptions,
    seed=0,
    progress=True,
    write_text=True,
)

In [ ]:
mappingOptions = frontend.MappingOptions(
    use_prior_position=USE_MAPPER_POSE_PRIORS,
    # A flat nadir block cannot determine focal length from imagery: focal
    # trades against scene depth. Hold the intrinsics at the known values;
    # the GCP bundle can free them later, where the control breaks the
    # degeneracy.
    refine_focal_length=False,
    # COLMAP's own tie-point filter defaults, restated so the tuning surface
    # is visible. All three are NO-OPS on this block - the 80/60 overlap
    # gives strong geometry: measured triangulation angle p1 = 7.4 deg,
    # median 26.8 deg, zero points below 5 deg, max reprojection 3.98 px -
    # but they matter on weaker blocks.
    filter_max_reproj_error=4.0,
    filter_min_tri_angle_deg=1.5,
    ba_local_min_tri_angle_deg=6.0,
)

rec = frontend.run_mapping(
    database_path=database_path,
    image_path=image_path,
    output_path=sfm_free_path,
    options=mappingOptions,
    seed=0,
    progress=True,
    write_text=True,
)

In [ ]:
utils.banner("Free network vs the known camera centres")
rmse = utils.camera_center_rmse(rec, camera_centers)
print(f"camera centre RMSE: {rmse:.4f} m")

# A bundle adjustment is a local optimiser: if the network sits in an arbitrary
# frame, the GCP residuals start hopeless and the solve diverges.
# A least-squares similarity over all 39 centres brings it close enough.
if rmse > 1.0:
    utils.align_to_positions(rec, camera_centers)
    print(f"after Sim3: {utils.camera_center_rmse(rec, camera_centers):.4f} m")

    # Overwrite the free network with the aligned version
    utils.export_reconstruction(rec, sfm_free_path)

# A similarity absorbs uniform scale but NOT anisotropic distortion, so a large
# residual that survives the alignment means the free network is deformed -
# almost always a runaway self-calibration - and no control will rescue it.

In [ ]:
# Scored before the control enters, so the effect of the GCPs is measurable.
# No adjusted marker coordinates exist yet, hence the empty dict: only the poses
# and the intrinsics can be scored at this point.
utils.block_report(rec)
utils.accuracy_report(rec, {}, gt, "free network")

## 3. GCP-controlled bundle adjustment

Each marker becomes a parameter block of its own, tied to the poses by its image
marks and - for the control points only - to the survey by a Gaussian prior. The
check points get rays but no prior, so their residual is the one unbiased
accuracy figure in the whole notebook.

All sigmas are given in native units (px, m); the backend expresses them in
tie-point sigma units, because COLMAP adds its tie-point residuals unweighted.

In [ ]:
solverOptions = backend.SolverOptions(
    refine_focal_length=REFINE_INTRINSICS,
    # Points with fewer rays would be IGNORED by the solver, not deleted:
    # they would keep pre-bundle coordinates and still reach the export.
    # 0 = disabled; the post-solve filter below is the one that deletes.
    min_track_length=0,
    # COLMAP's global-BA default. SOFT_L1 @ 1.0 would instead down-weight the
    # reprojection tail (p95 1.77, p99 2.27, max 3.98 px) without discarding
    # redundancy - the soft alternative to hard filtering.
    loss_function_type=None,
    # COLMAP ships function_tolerance = 0, which *disables* the relative
    # stopping test and makes every solve run to the iteration cap and report
    # NO_CONVERGENCE although the cost stopped improving. 1e-8 fixes it.
    function_tolerance=1e-8,
)
markersConstraints = backend.MarkerConstraints(
    observations=marker_data,
    xyz=surveyed_xyz,
    control_names=control_names,  # the check points stay unconstrained
    sigma_px=gcp_sigma_px,
    sigma_m=gcp_sigma_m,
    robust_loss_scale=2.8,  # Cauchy on the marks, to absorb mis-identification
)
camera_posesConstraints = (
    backend.CameraPoseConstraints(
        positions=camera_centers, sigma_m=camera_position_sigma_m
    )
    if USE_CAMERA_PRIORS
    else None
)
filterOptions = backend.FilterOptions(
    max_reproj_error=4.0, min_tri_angle_deg=1.5, min_track_length=0
)

result = backend.run_bundle_adjustment(
    rec,
    solver=solverOptions,
    markers=markersConstraints,
    camera_poses=camera_posesConstraints,
    # Post-solve filtering, same thresholds as the mapper: no-ops here, kept
    # explicit so every threshold in the notebook is visible in one place.
    filters=filterOptions,
    tie_points_sigma_px=tie_points_sigma_px,
    covariance=True,
    verbose=True,
)
marker_xyz = result.marker_xyz  # adjusted marker coordinates

In [ ]:
cov = result.covariance
cov.get_point_cov(1)

In [ ]:
utils.block_report(rec, marker_data, marker_xyz)
utils.accuracy_report(
    rec, marker_xyz, gt, "after GCP bundle", camera_priors_used=USE_CAMERA_PRIORS
)

## 4. Export

Binary + text model for COLMAP, plus a point cloud carrying track length and
reprojection error per point so the sparse cloud can be filtered in a viewer.

In [ ]:
utils.export_reconstruction(rec, sfm_gcp_path)
utils.save_point_cloud_as_ply(rec, sfm_gcp_path / "sparse_gcp_o3d.ply")